# Clusters

## All Imports

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
import scipy.cluster.hierarchy as shc
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import AgglomerativeClustering, KMeans
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from yellowbrick.cluster import KElbowVisualizer

In [ ]:
df = pd.read_csv("../data/pokedex_post_eda.csv")
df.info()

In [ ]:
df2 = pd.read_csv("../data/pokedex_description.csv")
df2.head()

## Cluster (HP, Attack, Defense, SP_Attack, SP_defense, Speed):

### Metrics

In [ ]:
colunas_batalha = ['HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed']
X = df[colunas_batalha]

scaler = PowerTransformer(method='yeo-johnson', standardize=True)
X_scaled = scaler.fit_transform(X)

plt.figure(figsize=(12, 8))
plt.title("Dendrograma de Pokémon (Baseado em Status de Batalha)")
plt.xlabel("Pokémons (Índice)")
plt.ylabel("Distância Euclidiana (Dissimilaridade)")

dend = shc.dendrogram(shc.linkage(X_scaled, method='ward'),
                      truncate_mode='lastp',
                      p=30,
                      leaf_rotation=90.,
                      leaf_font_size=10.,
                      show_contracted=True)

plt.axhline(y=17, color='r', linestyle='--') 
plt.show()

In [ ]:
range_n_clusters = [2, 3, 4, 5, 6, 7, 8, 9, 10]
silhouette_avg = []

for num_clusters in range_n_clusters:
    clusterer = AgglomerativeClustering(n_clusters=num_clusters)
    cluster_labels = clusterer.fit_predict(X_scaled)
    
    silhouette_avg.append(silhouette_score(X_scaled, cluster_labels))

plt.figure(figsize=(8, 4))
plt.plot(range_n_clusters, silhouette_avg, 'bx-')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.show()

#### Params

In [ ]:
n_clusters = 7

### Group

In [ ]:
colunas_stats = ['HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed']
X = df[colunas_stats]

pipeline = Pipeline([
    ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)),
    ('clustering', AgglomerativeClustering(n_clusters=n_clusters))
])

df['class'] = pipeline.fit_predict(X)

perfil_medio = df.groupby('class')[colunas_stats].mean()

contagem = df['class'].value_counts()

print("Média dos Status por Cluster:")
print(perfil_medio)
print("\nQuantidade de Pokémon por Cluster:")
print(contagem)

for i in range(n_clusters):
    print(f"\nExemplos do Cluster {i}:")
    print(df[df['class'] == i]['Name'].head(5).values)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(pipeline.named_steps['scaler'].transform(X))

df_plot = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_plot['Cluster'] = df['class']

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='Cluster', palette='viridis', s=60)
plt.title("Visualização dos Clusters de Pokémon (Reduzido via PCA)")
plt.xlabel("Componente Principal 1 (Geralmente reflete o 'Poder Total')")
plt.ylabel("Componente Principal 2 (Geralmente reflete o estilo 'Ofensivo vs Defensivo')")
plt.show()

In [ ]:
cross_cat = pd.crosstab(df['class'], df['Category'], normalize='index') * 100

for i in range(n_clusters):
    top_cat = cross_cat.loc[i].idxmax()
    prop_cat = cross_cat.loc[i].max()
    avg_get = df[df['class'] == i]['Get_Rate'].mean()
    
    n_legendaries = df[(df['class'] == i) & (df['Category'] == 'Legendary')].shape[0]
    n_megas = df[(df['class'] == i) & (df['Mega_Evolution_Flag'].notna())].shape[0]
    n_mythicals = df[(df['class'] == i) & (df['Category'] == 'Mythical')].shape[0]
    n_semi = df[(df['class'] == i) & (df['Category'] == 'Semi-Legendary')].shape[0]
    
    print(f"CLUSTER {i}:")
    print(f" - Taxa de Captura Média: {avg_get:.1f}")
    print(f" - Quantidade de Lendários: {n_legendaries}")
    print(f" - Quantidade de Míticos: {n_mythicals}")
    print(f" - Quantidade de Semi-Lendários: {n_semi}")
    print(f" - Quantidade de megas: {n_megas}")
    print("-" * 30)

In [ ]:
for i in range(n_clusters):
    print(f"--- Cluster {i} Samples ---")
    print(df[df['class'] == i]['Name'].sample(5).values)
    print("-"*30)

## Cluster Description

### Embeddings

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2') 

In [ ]:
embeddings = model.encode(df2["info"].tolist())
df2['embeddings'] = list(embeddings)

In [ ]:
df2.head()

In [ ]:
df2["embeddings"][0].size

In [ ]:
X = np.vstack(df2['embeddings'].values)

### K-Definition

In [ ]:
visualizer = KElbowVisualizer(KMeans(random_state=42), k=(2,15))
visualizer.fit(X) 
visualizer.show()

### Clusters

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X)

In [ ]:
df2["cluster"]=kmeans.labels_.astype(str)

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(X)

df2['pca_x'] = pca_result[:, 0]
df2['pca_y'] = pca_result[:, 1]

In [ ]:
centroides_original = kmeans.cluster_centers_
centroides_2d = pca.transform(centroides_original)

df_centroides = pd.DataFrame(centroides_2d, columns=['pca_x', 'pca_y'])
df_centroides['cluster'] = df_centroides.index.astype(str)

fig = px.scatter(
    df2, 
    x='pca_x', 
    y='pca_y', 
    color='cluster',
    hover_data=['name'],
    title='Clusters Pokémon + Centroides',
    opacity=0.6
)

fig.add_trace(
    go.Scatter(
        x=df_centroides['pca_x'],
        y=df_centroides['pca_y'],
        mode='markers',
        name='Centroides',
        marker=dict(
            color='black',
            size=10,
            symbol='x',
            line=dict(width=2)
        ),
        text=[f"Cluster Center {i}" for i in range(len(df_centroides))],
        hoverinfo='text'
    )
)

fig.show()

In [ ]:
for id in sorted(df2['cluster'].unique()):
    print(f"\nCLUSTER {id}:")
    group = df2[df2['cluster'] == id]
    
    sample = group.sample(n=min(7, len(group)), random_state=42)
    
    for name, description in zip(sample['name'], sample['info']):
        print(f"   - {name}: {description}")